# 48: python best practices

writing code that *works* is only step one - real-world code also has to be readable, maintainable, and safe
to change six months from now (often by someone else, or by a future you who's forgotten all the context).
this lesson collects the practices that matter most: clean code habits, the SOLID principles, a few classic
design patterns, and the anti-patterns that trip up beginners and experienced developers alike.

## clean code basics

* **naming**: a variable/function name should tell you what it IS or DOES without needing a comment.
  `d` tells you nothing, `elapsed_seconds` tells you everything
* **small functions, one job each**: a function that does one clearly-named thing is easy to test, easy to
  reuse, and easy to reason about. a 200-line function that does five things is none of those
* **DRY (Don't Repeat Yourself)**: if you've copy-pasted the same logic three times, that's a function/class
  waiting to be extracted - one bug fix should not require editing three places
* **single responsibility**: a class/function should have exactly one reason to change

let's look at a small, deliberately messy function and clean it up applying exactly these ideas.

In [1]:
# BEFORE: works, but violates several clean-code habits at once
def process(d):
    r = []
    for i in d:
        if i["status"] == "active" and i["balance"] > 0:
            t = i["balance"] * 0.05 if i["balance"] > 1000 else i["balance"] * 0.02
            r.append({"name": i["name"], "bonus": t})
    return r


# AFTER: same behavior, but every name says what it holds, and the "which rate applies" decision
# is its own small function - easy to test on its own, and to change later without touching the loop
def calculate_bonus_rate(balance):
    return 0.05 if balance > 1000 else 0.02


def is_eligible_for_bonus(customer):
    return customer["status"] == "active" and customer["balance"] > 0


def calculate_customer_bonuses(customers):
    return [
        {"name": customer["name"], "bonus": customer["balance"] * calculate_bonus_rate(customer["balance"])}
        for customer in customers
        if is_eligible_for_bonus(customer)
    ]


customers = [
    {"name": "Alice", "status": "active", "balance": 1500},
    {"name": "Bob", "status": "inactive", "balance": 500},
    {"name": "Carol", "status": "active", "balance": 200},
]
print("before-style result:", process(customers))
print("after-style result: ", calculate_customer_bonuses(customers))
print("\nsame output either way - but the second version is testable piece by piece and reads like english")

before-style result: [{'name': 'Alice', 'bonus': 75.0}, {'name': 'Carol', 'bonus': 4.0}]
after-style result:  [{'name': 'Alice', 'bonus': 75.0}, {'name': 'Carol', 'bonus': 4.0}]

same output either way - but the second version is testable piece by piece and reads like english


## SOLID, with tiny concrete examples

SOLID is five principles for keeping object-oriented code flexible as it grows (we first met classes/inheritance
back in lessons 07/08 - these principles are about how to *structure* that code well):

* **S - Single Responsibility**: a class should have one reason to change (just demonstrated above)
* **O - Open/Closed**: you should be able to ADD new behavior without MODIFYING existing, working code
* **L - Liskov Substitution**: a subclass should be usable anywhere its parent class is expected, without
  surprises
* **I - Interface Segregation**: don't force a class to implement methods it doesn't actually need
* **D - Dependency Inversion**: depend on an abstraction (an interface/protocol), not a specific concrete class

In [2]:
from typing import Protocol

# Open/Closed in action: adding a new discount type means adding a new CLASS, zero existing code changes
class DiscountStrategy(Protocol):
    def apply(self, price: float) -> float: ...


class NoDiscount:
    def apply(self, price):
        return price


class PercentageDiscount:
    def __init__(self, percent):
        self.percent = percent

    def apply(self, price):
        return price * (1 - self.percent / 100)


class FixedAmountDiscount:  # <-- adding this class is the ONLY change needed to support a new discount kind
    def __init__(self, amount):
        self.amount = amount

    def apply(self, price):
        return max(0, price - self.amount)


def checkout(price, discount: DiscountStrategy):
    return discount.apply(price)


print("no discount:       ", checkout(100, NoDiscount()))
print("20% off:            ", checkout(100, PercentageDiscount(20)))
print("$15 off:             ", checkout(100, FixedAmountDiscount(15)))

# Liskov Substitution violated: Square "is-a" Rectangle in real-world language, but forcing that inheritance
# breaks callers that reasonably expect setting width/height independently to work
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


class BadSquare(Rectangle):  # this looks reasonable, but it's an LSP trap
    def __init__(self, side):
        super().__init__(side, side)

    def set_width(self, width):
        self.width = self.height = width  # surprise! setting width silently changes height too


def resize_and_check(rect: Rectangle):
    rect.width = 4
    rect.height = 5
    return rect.area()  # any caller of this function reasonably expects 4 * 5 = 20


print("\nresize_and_check on a real Rectangle:", resize_and_check(Rectangle(1, 1)))
bad_square = BadSquare(1)
bad_square.width, bad_square.height = 4, 5  # directly setting attributes, bypassing the "helpful" setter
print("resize_and_check on a BadSquare:      ", resize_and_check(bad_square), "<- still correct here,")
print("but ANY code path that goes through set_width() breaks the width/height-are-independent assumption")
print("the fix: don't model Square as a Rectangle subclass at all - composition over inheritance (lesson 08)")

no discount:        100
20% off:             80.0
$15 off:              85

resize_and_check on a real Rectangle: 20
resize_and_check on a BadSquare:       20 <- still correct here,
but ANY code path that goes through set_width() breaks the width/height-are-independent assumption
the fix: don't model Square as a Rectangle subclass at all - composition over inheritance (lesson 08)


## common anti-patterns to watch out for

a few mistakes that are easy to make and genuinely bite people in production code:

In [3]:
# anti-pattern 1: THE mutable default argument trap - probably python's most famous gotcha
def add_item_broken(item, basket=[]):  # danger: this list is created ONCE, when the function is DEFINED,
    basket.append(item)                 # not once per call - every call without an explicit basket SHARES it
    return basket


print("call 1:", add_item_broken("apple"))
print("call 2:", add_item_broken("banana"), "<- 'apple' is still there! the same list got reused")


def add_item_fixed(item, basket=None):
    if basket is None:
        basket = []  # a fresh list, every single call
    basket.append(item)
    return basket


print("\nfixed call 1:", add_item_fixed("apple"))
print("fixed call 2:", add_item_fixed("banana"), "<- correctly starts empty each time")

# anti-pattern 2: bare `except:` swallows EVERYTHING, including typos and Ctrl+C (KeyboardInterrupt) -
# always catch the specific exception(s) you actually expect and know how to handle (recap lesson 06)
def risky_lookup_bad(data, key):
    try:
        return data[key]
    except:  # noqa - shown deliberately as the anti-pattern
        return None  # this also silently hides real bugs like a typo'd variable name elsewhere in the try block


def risky_lookup_good(data, key):
    try:
        return data[key]
    except KeyError:
        return None


print("\nbad version hides KeyError (fine here) but would ALSO hide a real bug the same way:",
      risky_lookup_bad({"a": 1}, "missing"))
print("good version is explicit about exactly what it's prepared to handle:",
      risky_lookup_good({"a": 1}, "missing"))

# anti-pattern 3: magic numbers - a raw number with no explanation of what it means or why that value
def is_adult_bad(age):
    return age >= 18  # 18 what? legal adulthood where? this reads fine ONCE, then rots

LEGAL_ADULT_AGE = 18  # now it's a single, named, searchable, changeable fact about the business rule

def is_adult_good(age):
    return age >= LEGAL_ADULT_AGE

print("\nboth work identically:", is_adult_bad(20), is_adult_good(20))
print("but only the second one is self-documenting and has exactly one place to update if the rule changes")

call 1: ['apple']
call 2: ['apple', 'banana'] <- 'apple' is still there! the same list got reused

fixed call 1: ['apple']
fixed call 2: ['banana'] <- correctly starts empty each time

bad version hides KeyError (fine here) but would ALSO hide a real bug the same way: None
good version is explicit about exactly what it's prepared to handle: None

both work identically: True True
but only the second one is self-documenting and has exactly one place to update if the rule changes


## a few classic design patterns

design patterns are named, reusable solutions to problems that come up again and again. we've actually already
used a couple without naming them (the `Protocol`-based discount classes above are the **Strategy** pattern).
here are three more, each in its simplest possible python form.

In [4]:
# Factory pattern: centralize the "which class do I need to build" decision in one place,
# so callers just ask for what they want by name and never need to know the concrete class names at all
class EmailNotifier:
    def send(self, message):
        return f"[email] {message}"


class SMSNotifier:
    def send(self, message):
        return f"[sms] {message}"


def notifier_factory(kind):
    notifiers = {"email": EmailNotifier, "sms": SMSNotifier}
    if kind not in notifiers:
        raise ValueError(f"unknown notifier kind: {kind}")
    return notifiers[kind]()


for kind in ("email", "sms"):
    notifier = notifier_factory(kind)
    print(notifier.send("your order has shipped"))

[email] your order has shipped
[sms] your order has shipped


In [5]:
# Observer pattern: one "subject" holds a list of interested "observers" and notifies ALL of them whenever
# something happens - the subject doesn't need to know anything about what each observer actually does with it
class OrderSubject:
    def __init__(self):
        self._observers = []

    def subscribe(self, observer):
        self._observers.append(observer)

    def notify_all(self, order):
        for observer in self._observers:
            observer(order)  # each observer is just a callable here - functions work fine as observers too


def log_order(order):
    print(f"[logger] new order: {order}")


def send_confirmation_email(order):
    print(f"[email] confirmation sent for: {order}")


subject = OrderSubject()
subject.subscribe(log_order)
subject.subscribe(send_confirmation_email)
subject.notify_all("order #1042")
print("\nnote: subject.notify_all() knows NOTHING about logging or email - adding a 3rd observer later")
print("(sms alerts, analytics, whatever) needs zero changes to OrderSubject itself - that's Open/Closed again")

[logger] new order: order #1042
[email] confirmation sent for: order #1042

note: subject.notify_all() knows NOTHING about logging or email - adding a 3rd observer later
(sms alerts, analytics, whatever) needs zero changes to OrderSubject itself - that's Open/Closed again


## testing, logging, and project structure - quick recap with a "why it's a best practice" lens

these were all covered as *features* earlier in the course - here's the best-practice framing for each:

* **write tests, and write them small** (lesson 15's `unittest`): a test suite that runs in seconds gives you
  the confidence to refactor fearlessly. the "test pyramid" idea: have LOTS of fast small unit tests, fewer
  slower integration tests, and very few full end-to-end tests - inverting that (mostly slow end-to-end tests)
  is a common real-world mistake that makes the test suite too slow to actually run often
* **use `logging`, never leave `print()` in production code** (also lesson 15): logging has levels
  (DEBUG/INFO/WARNING/ERROR), can be turned up or down without editing code, and can be routed to a file or a
  monitoring system - a stray `print()` has none of that and is easy to forget and leave behind
* **use a virtual environment and pin your dependencies** (lesson 00, `requirements.txt`/`pyproject.toml` from
  lesson 16): "works on my machine" is not good enough - reproducible installs matter the moment more than one
  person (or one deployment server) runs the code
* **type hints on public functions** (lesson 10): they're free documentation that a type checker can actually
  verify for you, catching a whole category of bugs before the code even runs

## a short code review checklist

when reviewing your own code (or a teammate's) before it ships, this is a genuinely useful pass to run through:

1. does every function/variable name say what it holds or does, with no extra explanation needed?
2. is there any duplicated logic that should be one shared function instead?
3. are exceptions caught specifically, or is there a bare `except:` hiding something?
4. any mutable default arguments (`def f(x=[])`)?
5. any magic numbers/strings that should be a named constant?
6. is there a test covering the new behavior, especially the edge cases?
7. could a class here be replaced by a plain function, or vice versa - are we over- or under-engineering it?

## quick comparison to other languages

* enforcing SOLID and clean interfaces is often more STRICT elsewhere: Java/C# make you explicitly declare
  `interface`/`abstract class` and the compiler enforces it; Go has no inheritance at all, and leans entirely
  on small interfaces (satisfied implicitly - no `implements` keyword needed) to get the same Dependency
  Inversion benefit; Rust's `trait` system is similar to Go's interfaces but checked even more strictly at
  compile time
* the *design patterns* themselves (Strategy/Factory/Observer/etc.) come from the famous 1994 "Gang of Four"
  book and are genuinely language-agnostic - you'll recognize the exact same shapes in Java, C#, C++ and beyond,
  they're just usually more verbose there because those languages require declaring an explicit interface first
* the ecosystem of *automated* best-practice enforcement differs a lot: JavaScript/TypeScript leans on
  ESLint + Prettier, Java on Checkstyle/SpotBugs, C#/.NET on Roslyn analyzers, Rust on `clippy` (famously strict
  and helpful), Go on `go vet`/`golangci-lint` - python's equivalents are tools like `ruff`, `black`, `mypy`
  and `pylint`, worth adopting on any real project

## exercises

1. take this function and refactor it following the clean-code ideas above (small single-purpose helpers,
   clear names, no magic numbers):
   ```python
   def calc(o):
       t = 0
       for i in o:
           if i[1] > 10:
               t += i[1] * 0.9
           else:
               t += i[1]
       return t
   ```
   (each `i` is a `(name, quantity)` tuple; a 10% discount applies once quantity is over 10)
2. fix this mutable-default-argument bug:
   ```python
   def add_tag(tag, tags=[]):
       tags.append(tag)
       return tags
   ```
3. implement a second `DiscountStrategy` class from the Strategy-pattern section above: a
   `BuyOneGetOneFreeDiscount` that, given a `price` and an item `quantity`, returns the total price for that
   quantity where every 2nd item is free (assume `apply(self, price, quantity)` this time, an extra parameter)
4. add a third observer function to the `OrderSubject` example that prints an SMS-style alert, and subscribe
   it - without changing `OrderSubject` itself at all

In [6]:
# TODO: exercise 1 - refactor calc()
# TODO: exercise 2 - fix add_tag()
# TODO: exercise 3 - BuyOneGetOneFreeDiscount
# TODO: exercise 4 - a third observer

In [7]:
# sample solution 1
BULK_DISCOUNT_THRESHOLD = 10
BULK_DISCOUNT_RATE = 0.9

def line_total(quantity):
    return quantity * BULK_DISCOUNT_RATE if quantity > BULK_DISCOUNT_THRESHOLD else quantity

def calculate_order_total(order_lines):
    return sum(line_total(quantity) for _name, quantity in order_lines)

order = [("widget", 5), ("gadget", 15)]
print("exercise 1 - order total:", calculate_order_total(order))

# sample solution 2
def add_tag(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags

print("\nexercise 2 - call 1:", add_tag("red"))
print("exercise 2 - call 2:", add_tag("blue"), "<- correctly starts fresh")

# sample solution 3
class BuyOneGetOneFreeDiscount:
    def apply(self, price, quantity):
        free_items = quantity // 2
        paid_items = quantity - free_items
        return price * paid_items

bogo = BuyOneGetOneFreeDiscount()
print("\nexercise 3 - 5 items at $10 each, BOGO applied:", bogo.apply(10, 5))

# sample solution 4
def send_sms_alert(order):
    print(f"[sms] alert! {order}")

subject.subscribe(send_sms_alert)  # reusing the OrderSubject instance from earlier - ZERO changes to its class
subject.notify_all("order #1043")

exercise 1 - order total: 18.5

exercise 2 - call 1: ['red']
exercise 2 - call 2: ['blue'] <- correctly starts fresh

exercise 3 - 5 items at $10 each, BOGO applied: 30
[logger] new order: order #1043
[email] confirmation sent for: order #1043
[sms] alert! order #1043


## what we learned

* clean-code habits (naming, small single-purpose functions, DRY) turn "works" into "works AND is maintainable"
* the SOLID principles, and concretely what it looks like to follow (Open/Closed via Strategy) or violate
  (Liskov Substitution via a naive Square-extends-Rectangle) one of them
* three classic design patterns - Strategy, Factory, Observer - each solving a recurring structural problem in
  a way that's completely language-agnostic
* real anti-patterns that catch even experienced developers off guard: the mutable default argument trap, bare
  `except:`, and magic numbers
* a concrete checklist to run through before calling any piece of code "done"

this closes out the "self-implementations and best practices" category - lesson 47 built core data structures
from scratch to understand how the built-ins work, and this lesson turned toward writing code that stays
readable and flexible as it grows.